In [ ]:
!pip install -q lightgbm rapidfuzz pyarrow

In [ ]:
import os
os.makedirs('src', exist_ok=True)
open('src/__init__.py', 'w').close()
print('src/ directory and __init__.py created.')

In [ ]:
%%writefile src/normalize.py
"""
Vectorized text normalization for business names and addresses.

All public functions work on entire pandas Series (not row-by-row)
so they run in seconds on millions of records.
"""

import re
import unicodedata
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Legal suffix maps — sorted longest-first for greedy matching
# ---------------------------------------------------------------------------
_LEGAL_SUFFIX_MAP = {
    # English multi-word (must come before single-word variants)
    "private limited": "pvt ltd",
    "pvt limited": "pvt ltd",
    "pvt. limited": "pvt ltd",
    "pvt. ltd.": "pvt ltd",
    "pvt. ltd": "pvt ltd",
    "pvt ltd": "pvt ltd",
    "p. ltd.": "pvt ltd",
    "p ltd": "pvt ltd",
    "limited liability partnership": "llp",
    "limited liability company": "llc",
    "incorporated": "inc",
    "incorporation": "inc",
    "corporation": "corp",
    "limited": "ltd",
    "company": "co",
    # French (accent-stripped versions, so ç → c, é → e etc.)
    "societe a responsabilite limitee": "sarl",
    "societe par actions simplifiee unipersonnelle": "sasu",
    "societe par actions simplifiee": "sas",
    "societe anonyme": "sa",
    "entreprise unipersonnelle a responsabilite limitee": "eurl",
    "societe en nom collectif": "snc",
    "groupement d interet economique": "gie",
    "societe civile immobiliere": "sci",
    "societe civile": "sc",
    # Abbreviation forms (already canonical)
    "sarl": "sarl",
    "sasu": "sasu",
    "sas": "sas",
    "eurl": "eurl",
    "snc": "snc",
    "gie": "gie",
    "sci": "sci",
    "llp": "llp",
    "llc": "llc",
    "inc": "inc",
    "corp": "corp",
    "ltd": "ltd",
}

# Build patterns sorted longest → shortest
_SUFFIX_KEYS = sorted(_LEGAL_SUFFIX_MAP.keys(), key=len, reverse=True)

# Single combined regex: captures the first matching suffix
# \b word boundary on both sides so "ltd" in "ltd company" matches correctly
_SUFFIX_RE_STR = r"(?:^|(?<=\s))(" + "|".join(re.escape(k) for k in _SUFFIX_KEYS) + r")(?:\s|$)"
_SUFFIX_RE = re.compile(_SUFFIX_RE_STR)

# Reversed map: canonical suffix → set of raw forms (not needed but kept for reference)
# Replacement map (what to substitute the match with)
_SUFFIX_REPLACE = {k: _LEGAL_SUFFIX_MAP[k] for k in _SUFFIX_KEYS}

# ---------------------------------------------------------------------------
# Address abbreviation map (applied with word-boundary regex)
# ---------------------------------------------------------------------------
_ADDR_ABBREVS = {
    r"\bst\b": "street",
    r"\bave\b": "avenue",
    r"\brd\b": "road",
    r"\bblvd\b": "boulevard",
    r"\bbd\b": "boulevard",
    r"\bdr\b": "drive",
    r"\bln\b": "lane",
    r"\bct\b": "court",
    r"\bpl\b": "place",
    r"\bsq\b": "square",
    r"\bpkwy\b": "parkway",
    r"\bhwy\b": "highway",
    r"\bfwy\b": "freeway",
    r"\bexpy\b": "expressway",
}
# Combined single-pass regex for address abbreviations
_ADDR_RE = re.compile("|".join(f"(?P<g{i}>{pat})" for i, pat in enumerate(_ADDR_ABBREVS)))
_ADDR_FULL = list(_ADDR_ABBREVS.values())
_ADDR_PATS = list(_ADDR_ABBREVS.keys())


# ---------------------------------------------------------------------------
# Vectorized helpers
# ---------------------------------------------------------------------------

def _strip_accents_series(s: pd.Series) -> pd.Series:
    """NFKD normalization + remove diacritics, vectorized via unicodedata."""
    def _strip(text):
        if not isinstance(text, str):
            return ""
        nfkd = unicodedata.normalize("NFKD", text)
        return "".join(c for c in nfkd if unicodedata.category(c) != "Mn")
    return s.apply(_strip)


def normalize_name_series(names: pd.Series):
    """
    Vectorized normalize_name for an entire Series.
    Returns tuple of three Series: (core_name, suffix, full_norm).
    """
    # Convert nan → ""
    text = names.fillna("").astype(str)

    # Strip accents
    text = _strip_accents_series(text)
    # Lowercase
    text = text.str.lower()
    # & → and
    text = text.str.replace(r"&", " and ", regex=False)
    # Remove punctuation except spaces/hyphens-between-words
    text = text.str.replace(r"[^\w\s]", " ", regex=True)
    # Collapse whitespace
    text = text.str.replace(r"\s+", " ", regex=True).str.strip()

    # Suffix extraction (vectorized via apply on each element — unavoidable
    # because suffix removal modifies the text differently per row)
    def _extract_suffix(t):
        m = _SUFFIX_RE.search(t)
        if m:
            matched = m.group(1)
            canonical = _LEGAL_SUFFIX_MAP.get(matched, matched)
            # Remove the matched suffix from the text
            core = (t[: m.start()].strip() + " " + t[m.end() :].strip()).strip()
            core = re.sub(r"\s+", " ", core).strip()
            return core, canonical
        return t, ""

    results = text.apply(_extract_suffix)
    core   = results.apply(lambda x: x[0])
    suffix = results.apply(lambda x: x[1])
    full   = core + suffix.apply(lambda s: (" " + s) if s else "")
    full   = full.str.strip()

    return core, suffix, full


def normalize_address_series(addresses: pd.Series):
    """
    Vectorized normalize_address.
    Returns tuple of two Series: (norm_addr, numeric_tokens list).
    """
    addr = addresses.fillna("").astype(str)
    # Replace nan / null strings
    addr = addr.str.replace(r"^(nan|null|none)$", "", regex=True)

    # Strip accents (vectorized)
    addr = _strip_accents_series(addr)

    # Lowercase
    addr = addr.str.lower()

    # Expand abbreviations (compiled regex, single pass per row via str.replace loops)
    # This is O(n_abbrevs) passes but each pass is vectorized
    for pat, full in _ADDR_ABBREVS.items():
        addr = addr.str.replace(pat, full, regex=True)

    # Remove punctuation
    addr = addr.str.replace(r"[,./\-#()\"']", " ", regex=True)
    addr = addr.str.replace(r"\s+", " ", regex=True).str.strip()

    # Numeric token extraction — vectorized via findall
    def _extract_nums(text):
        if not text:
            return []
        nums = re.findall(r"\b\d+\b", text)
        return list({n for n in nums if len(n) >= 2})   # skip single digits

    numeric_tokens = addr.apply(_extract_nums)

    return addr, numeric_tokens


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add normalized columns to a dataframe containing 'business_name' and
    'business_address'. Adds:
        norm_core, norm_suffix, norm_name  — name columns
        norm_addr, numeric_tokens          — address columns
        is_non_latin                       — flag for non-ASCII name
    """
    df = df.copy()

    core, suffix, full = normalize_name_series(df["business_name"])
    df["norm_core"]   = core
    df["norm_suffix"] = suffix
    df["norm_name"]   = full

    norm_addr, num_toks = normalize_address_series(df["business_address"])
    df["norm_addr"]       = norm_addr
    df["numeric_tokens"]  = num_toks

    # Non-Latin flag (S1 is always Latin; S2/S3 may have Hindi/Tamil/etc.)
    df["is_non_latin"] = df["business_name"].apply(
        lambda x: bool(re.search(r"[^\x00-\x7F]", str(x)))
    )

    return df

In [ ]:
%%writefile src/blocking.py
"""
Blocking / candidate generation.

Memory-safe sharding strategy:
  - TF-IDF vectorizer is FIT once on full corpus (cheap — just vocabulary)
  - S1 is transformed once (small matrix, e.g. 300K × 30K)
  - S23 is transformed and scored in SHARDS of S23_SHARD_SIZE rows
    so peak sparse matrix RAM = shard_size × MAX_FEATURES (not full 5M × 80K)
  - Three passes: name TF-IDF + address TF-IDF + numeric exact match
  - Results merged across shards, then capped at MAX_TOTAL per S1
"""

import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(__file__)))

import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
TOP_K_NAME        = 20      # name TF-IDF top-k per S1 (per source)
TOP_K_ADDR        = 20      # address TF-IDF top-k per S1
MAX_NUMERIC       = 30      # max numeric-match candidates per S1
MAX_TOTAL         = 100     # hard cap on TOTAL candidates per S1 (across S2+S3)
MAX_FEATURES      = 30_000  # reduced from 80K → smaller sparse matrices
MIN_DF            = 3       # raised from 2 → smaller vocabulary
S23_SHARD_SIZE    = 500_000 # transform S23 in shards to limit peak RAM
BATCH_SIZE        = 2_000   # S1 rows per batch for dot product
MIN_TFIDF_SCORE   = 0.05
MIN_NUMERIC_LEN   = 3
MAX_NUMERIC_DOCFREQ = 500


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _fit_tfidf(corpus_texts, max_features=MAX_FEATURES):
    """Fit TF-IDF vectorizer on corpus_texts. Returns fitted vectorizer."""
    vect = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        max_features=max_features,
        min_df=MIN_DF,
        sublinear_tf=True,
        dtype=np.float32,
    )
    vect.fit(corpus_texts)
    return vect


def _sparse_topk_batch(s1_csr, s23_shard_csr, k, s23_offset, batch_size=BATCH_SIZE):
    """
    Dot-product top-k for one S23 shard.
    Returns list of (s1_i, s23_global_j, score).
    s23_offset: index offset to convert shard-local j to global j.
    """
    n_s1 = s1_csr.shape[0]
    results = []

    for start in range(0, n_s1, batch_size):
        end   = min(start + batch_size, n_s1)
        batch = s1_csr[start:end]
        scores = batch.dot(s23_shard_csr.T).tocsr()

        for local_i in range(scores.shape[0]):
            rs = scores.indptr[local_i]
            re = scores.indptr[local_i + 1]
            cols = scores.indices[rs:re]
            vals = scores.data[rs:re]

            mask = vals >= MIN_TFIDF_SCORE
            cols, vals = cols[mask], vals[mask]
            if len(cols) == 0:
                continue

            if len(cols) > k:
                top_idx = np.argpartition(-vals, k)[:k]
                cols, vals = cols[top_idx], vals[top_idx]

            s1_global = start + local_i
            for c, v in zip(cols, vals):
                results.append((s1_global, int(c) + s23_offset, float(v)))

    return results


def _tfidf_pass_sharded(s1_df, s23_df, field, k):
    """
    TF-IDF blocking pass, processing S23 in memory-safe shards.

    Fits vectorizer on full corpus (cheap), transforms S1 once,
    then transforms S23 in chunks of S23_SHARD_SIZE rows.

    Returns list of (s1_idx, s23_idx, score) — s23_idx is global row index.
    """
    s1_text  = s1_df[field].fillna("").astype(str).tolist()
    s23_text = s23_df[field].fillna("").astype(str).tolist()

    # Fit on full corpus — just builds vocabulary, not matrices
    vect = _fit_tfidf(s1_text + s23_text)

    # Transform S1 once (small)
    s1_mat = sk_normalize(vect.transform(s1_text), norm="l2").tocsr()

    # Per-(s1_i) accumulate best-k across all shards
    best_per_s1 = defaultdict(list)   # s1_i → [(s23_global_j, score)]

    n_s23 = len(s23_text)
    for shard_start in range(0, n_s23, S23_SHARD_SIZE):
        shard_end  = min(shard_start + S23_SHARD_SIZE, n_s23)
        shard_text = s23_text[shard_start:shard_end]

        shard_mat = sk_normalize(vect.transform(shard_text), norm="l2").tocsr()
        triples   = _sparse_topk_batch(s1_mat, shard_mat, k, shard_start)

        for s1_i, s23_j, sc in triples:
            best_per_s1[s1_i].append((s23_j, sc))

        del shard_mat  # free shard matrix immediately

    # Merge: keep global top-k per S1
    results = []
    for s1_i, pairs in best_per_s1.items():
        if len(pairs) > k:
            pairs.sort(key=lambda x: -x[1])
            pairs = pairs[:k]
        for s23_j, sc in pairs:
            results.append((s1_i, s23_j, sc))

    return results


def _numeric_pass(s1_df, s23_df, max_per_s1=MAX_NUMERIC,
                  min_len=MIN_NUMERIC_LEN, max_df=MAX_NUMERIC_DOCFREQ):
    """
    Exact numeric-token match with inverted index.
    Only uses tokens that are long enough and rare enough to be discriminative.
    Returns list of (s1_idx, s23_idx, token_count).
    """
    doc_freq = defaultdict(int)
    for tokens in s23_df["numeric_tokens"]:
        for tok in tokens:
            if len(tok) >= min_len:
                doc_freq[tok] += 1

    inv_index = defaultdict(list)
    for j, tokens in enumerate(s23_df["numeric_tokens"]):
        for tok in tokens:
            if len(tok) >= min_len and doc_freq[tok] <= max_df:
                inv_index[tok].append(j)

    results = []
    for i, tokens in enumerate(s1_df["numeric_tokens"]):
        hit_counts = defaultdict(int)
        for tok in tokens:
            if len(tok) >= min_len and doc_freq.get(tok, 0) <= max_df:
                for j in inv_index.get(tok, []):
                    hit_counts[j] += 1

        sorted_hits = sorted(hit_counts.items(), key=lambda x: -x[1])
        for j, cnt in sorted_hits[:max_per_s1]:
            results.append((i, j, cnt))

    return results


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def generate_candidates(
    s1_df: pd.DataFrame,
    s2_df: pd.DataFrame,
    s3_df: pd.DataFrame,
    top_k_name: int = TOP_K_NAME,
    top_k_addr: int = TOP_K_ADDR,
    verbose: bool = True,
    gt_pairs: set = None,
) -> pd.DataFrame:
    """
    Produce candidate (S1, S2/S3) pairs for scoring.

    Memory-safe: S23 TF-IDF transform is sharded so peak matrix RAM
    = S23_SHARD_SIZE × MAX_FEATURES (not full corpus size).

    Parameters
    ----------
    s1_df, s2_df, s3_df : normalized DataFrames with entity_id, country,
        norm_name, norm_addr, numeric_tokens columns.
    gt_pairs : optional set of (s1_id, s23_id) for per-pass recall logging.

    Returns
    -------
    DataFrame: source1_entity_id, candidate_id, name_score, addr_score,
               numeric_match, source
    """
    all_pairs = []
    countries = s1_df["country"].unique()

    after_name_pairs = set()
    after_addr_pairs = set()
    after_num_pairs  = set()

    for country in countries:
        if verbose:
            print(f"  [blocking] country={country}", flush=True)

        s1_c = s1_df[s1_df["country"] == country].reset_index(drop=True)
        s2_c = s2_df[s2_df["country"] == country].reset_index(drop=True)
        s3_c = s3_df[s3_df["country"] == country].reset_index(drop=True)

        if len(s1_c) == 0:
            continue

        source_results = {}

        for src_name, s23_c in [("S2", s2_c), ("S3", s3_c)]:
            if len(s23_c) == 0:
                continue
            if verbose:
                print(f"    {src_name}: {len(s1_c):,} S1 × {len(s23_c):,} {src_name}", flush=True)

            name_triples = _tfidf_pass_sharded(s1_c, s23_c, "norm_name", top_k_name)
            addr_triples = _tfidf_pass_sharded(s1_c, s23_c, "norm_addr", top_k_addr)
            num_triples  = _numeric_pass(s1_c, s23_c)

            name_map = {(i, j): sc  for i, j, sc  in name_triples}
            addr_map = {(i, j): sc  for i, j, sc  in addr_triples}
            num_map  = {(i, j): cnt for i, j, cnt in num_triples}

            # Per-pass recall tracking
            if gt_pairs is not None:
                for (i, j) in name_map:
                    after_name_pairs.add((s1_c.iloc[i]["entity_id"], s23_c.iloc[j]["entity_id"]))
                for (i, j) in set(name_map) | set(addr_map):
                    after_addr_pairs.add((s1_c.iloc[i]["entity_id"], s23_c.iloc[j]["entity_id"]))
                for (i, j) in set(name_map) | set(addr_map) | set(num_map):
                    after_num_pairs.add((s1_c.iloc[i]["entity_id"], s23_c.iloc[j]["entity_id"]))

            all_keys = set(name_map) | set(addr_map) | set(num_map)

            pair_info = {}
            for (i, j) in all_keys:
                n_sc     = name_map.get((i, j), 0.0)
                a_sc     = addr_map.get((i, j), 0.0)
                n_cnt    = num_map.get((i, j), 0)
                combined = n_sc + a_sc + min(n_cnt, 5) * 0.1
                s23_id   = s23_c.iloc[j]["entity_id"]
                pair_info[(i, s23_id)] = (n_sc, a_sc, int((i, j) in num_map), combined, src_name)

            source_results[src_name] = pair_info

        # Merge S2 + S3, apply global cap per S1
        s1_buckets = defaultdict(list)
        for src_name, pair_info in source_results.items():
            for (i, s23_id), (n_sc, a_sc, num, combined, src) in pair_info.items():
                s1_buckets[i].append((s23_id, n_sc, a_sc, num, combined, src))

        for i, cand_list in s1_buckets.items():
            cand_list.sort(key=lambda x: -x[4])
            cand_list = cand_list[:MAX_TOTAL]
            s1_id = s1_c.iloc[i]["entity_id"]
            for (s23_id, n_sc, a_sc, num, _combined, src) in cand_list:
                all_pairs.append({
                    "source1_entity_id": s1_id,
                    "candidate_id":      s23_id,
                    "name_score":        n_sc,
                    "addr_score":        a_sc,
                    "numeric_match":     num,
                    "source":            src,
                })

    # Per-pass recall summary
    if gt_pairs is not None and verbose and gt_pairs:
        total = len(gt_pairs)
        def _r(pairs):
            n = len(gt_pairs & pairs)
            return f"{n}/{total} = {n/total:.4f}"
        print(f"\n  === Blocking Recall by Pass ===")
        print(f"  After name  TF-IDF : {_r(after_name_pairs)}")
        print(f"  After + addr TF-IDF: {_r(after_addr_pairs)}")
        print(f"  After + numeric    : {_r(after_num_pairs)}")
        print(f"  MAX_TOTAL={MAX_TOTAL}  shard={S23_SHARD_SIZE:,}  features={MAX_FEATURES:,}")

    if not all_pairs:
        return pd.DataFrame(columns=[
            "source1_entity_id", "candidate_id",
            "name_score", "addr_score", "numeric_match", "source"
        ])

    df = pd.DataFrame(all_pairs)
    df["_combined"] = df["name_score"] + df["addr_score"] + df["numeric_match"].astype(float)
    df = (df.sort_values("_combined", ascending=False)
            .drop_duplicates(subset=["source1_entity_id", "candidate_id"])
            .drop(columns=["_combined"])
            .reset_index(drop=True))
    return df

In [ ]:
%%writefile src/features.py
"""
Vectorized pair-level feature engineering.

Strategy: merge all required columns from S1 and S23 into the candidates
DataFrame in two fast hash-joins, then compute all features on aligned
numpy arrays.  This avoids per-row Python overhead even at 85M+ pairs.
"""

import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(__file__)))

import numpy as np
import pandas as pd
from rapidfuzz.distance import JaroWinkler, Levenshtein
from rapidfuzz.fuzz import token_set_ratio, token_sort_ratio


# ---------------------------------------------------------------------------
# Low-level vectorized helpers (list comprehensions over numpy object arrays)
# ---------------------------------------------------------------------------

def _jw(a, b):
    return np.fromiter(
        (JaroWinkler.normalized_similarity(x, y) for x, y in zip(a, b)),
        dtype=np.float32, count=len(a))

def _lev(a, b):
    return np.fromiter(
        (1.0 - Levenshtein.normalized_distance(x, y) for x, y in zip(a, b)),
        dtype=np.float32, count=len(a))

def _tsr(a, b):
    return np.fromiter(
        (token_set_ratio(x, y) / 100.0 for x, y in zip(a, b)),
        dtype=np.float32, count=len(a))

def _tsort(a, b):
    return np.fromiter(
        (token_sort_ratio(x, y) / 100.0 for x, y in zip(a, b)),
        dtype=np.float32, count=len(a))

def _c3j(a, b):
    def _ng(s, n=3):
        return set(s[i:i+n] for i in range(len(s)-n+1)) if len(s) >= n else set()
    def _j(x, y):
        sa, sb = _ng(x), _ng(y)
        if not sa and not sb: return 1.0
        if not sa or not sb:  return 0.0
        return len(sa & sb) / len(sa | sb)
    return np.fromiter((_j(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))

def _c4j(a, b):
    def _ng(s, n=4):
        return set(s[i:i+n] for i in range(len(s)-n+1)) if len(s) >= n else set()
    def _j(x, y):
        sa, sb = _ng(x), _ng(y)
        if not sa and not sb: return 1.0
        if not sa or not sb:  return 0.0
        return len(sa & sb) / len(sa | sb)
    return np.fromiter((_j(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))

def _tokj(a, b):
    def _j(x, y):
        sa = set(x.split()); sb = set(y.split())
        if not sa and not sb: return 1.0
        if not sa or not sb:  return 0.0
        return len(sa & sb) / len(sa | sb)
    return np.fromiter((_j(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))

def _first_tok_eq(a, b):
    def _eq(x, y):
        tx = x.split(); ty = y.split()
        return int(bool(tx and ty and tx[0] == ty[0]))
    return np.fromiter((_eq(x, y) for x, y in zip(a, b)), dtype=np.int8, count=len(a))

def _numj(a, b):
    def _j(x, y):
        sx = set(x); sy = set(y)
        if not sx and not sy: return 1.0
        if not sx or not sy:  return 0.0
        return len(sx & sy) / len(sx | sy)
    return np.fromiter((_j(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))

def _numovl(a, b):
    return np.fromiter(
        (int(bool(set(x) & set(y))) for x, y in zip(a, b)),
        dtype=np.int8, count=len(a))

def _zipm(a, b):
    def _z(x, y):
        zx = {n for n in x if len(n) in (5, 6)}
        zy = {n for n in y if len(n) in (5, 6)}
        return int(bool(zx & zy))
    return np.fromiter((_z(x, y) for x, y in zip(a, b)), dtype=np.int8, count=len(a))


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_features(
    candidates: pd.DataFrame,
    s1_df: pd.DataFrame,
    s23_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compute pair features for all candidate pairs.

    Uses two pandas hash-joins to pull S1/S23 columns into the candidates
    DataFrame before computing features — O(n_pairs) with minimal Python
    overhead.

    Parameters
    ----------
    candidates : DataFrame with columns
        source1_entity_id, candidate_id, name_score, addr_score,
        numeric_match, source
    s1_df, s23_df : normalized DataFrames (entity_id column required)

    Returns
    -------
    feature_df : DataFrame with FEATURE_COLS columns (same index as candidates)
    """
    S1_COLS  = ["entity_id", "norm_name", "norm_core", "norm_suffix",
                "norm_addr", "numeric_tokens"]
    S23_COLS = ["entity_id", "norm_name", "norm_core", "norm_suffix",
                "norm_addr", "numeric_tokens", "is_non_latin"]

    # Two hash-joins to pull all columns
    df = candidates.copy().reset_index(drop=True)
    df = df.merge(s1_df[S1_COLS], left_on="source1_entity_id",
                  right_on="entity_id", how="left",
                  suffixes=("", "_s1")).drop(columns=["entity_id_s1"] if "entity_id_s1" in df.columns else [])
    df = df.rename(columns={"norm_name":     "s1_norm_name",
                              "norm_core":     "s1_norm_core",
                              "norm_suffix":   "s1_norm_suffix",
                              "norm_addr":     "s1_norm_addr",
                              "numeric_tokens":"s1_nums"})
    df = df.merge(s23_df[S23_COLS], left_on="candidate_id",
                  right_on="entity_id", how="left",
                  suffixes=("", "_cnd")).drop(columns=["entity_id"] if "entity_id" in df.columns else [])
    df = df.rename(columns={"norm_name":     "cnd_norm_name",
                              "norm_core":     "cnd_norm_core",
                              "norm_suffix":   "cnd_norm_suffix",
                              "norm_addr":     "cnd_norm_addr",
                              "numeric_tokens":"cnd_nums",
                              "is_non_latin":  "cnd_non_latin"})

    # Extract aligned arrays
    s1_name  = df["s1_norm_name"].fillna("").values
    cnd_name = df["cnd_norm_name"].fillna("").values
    s1_core  = df["s1_norm_core"].fillna("").values
    cnd_core = df["cnd_norm_core"].fillna("").values
    s1_suf   = df["s1_norm_suffix"].fillna("").values
    cnd_suf  = df["cnd_norm_suffix"].fillna("").values
    s1_addr  = df["s1_norm_addr"].fillna("").values
    cnd_addr = df["cnd_norm_addr"].fillna("").values
    s1_nums  = [x if isinstance(x, list) else [] for x in df["s1_nums"]]
    cnd_nums = [x if isinstance(x, list) else [] for x in df["cnd_nums"]]
    non_lat  = df["cnd_non_latin"].fillna(False).astype(int).values

    # Suffix features
    n = len(df)
    suf_conflict = np.array(
        [(1 if (bool(a) and bool(b) and a != b) else 0) for a, b in zip(s1_suf, cnd_suf)],
        dtype=np.int8)
    suf_match = np.array(
        [(1 if (bool(a) and a == b) else 0) for a, b in zip(s1_suf, cnd_suf)],
        dtype=np.int8)

    # Blocking scores
    name_score = df["name_score"].values.astype(np.float32)
    addr_score = df["addr_score"].values.astype(np.float32)
    blk_num    = df["numeric_match"].values.astype(np.int8)
    is_s2      = (df["source"] == "S2").astype(np.int8).values
    combined   = name_score + addr_score + blk_num.astype(np.float32) * 0.5

    feat = pd.DataFrame({
        # Name
        "jaro_winkler":     _jw(s1_name, cnd_name),
        "levenshtein":      _lev(s1_name, cnd_name),
        "token_set_ratio":  _tsr(s1_name, cnd_name),
        "token_sort_ratio": _tsort(s1_name, cnd_name),
        "char3_jaccard":    _c3j(s1_core, cnd_core),
        "char4_jaccard":    _c4j(s1_core, cnd_core),
        "token_jaccard":    _tokj(s1_core, cnd_core),
        "first_token_eq":   _first_tok_eq(s1_core, cnd_core),
        "suffix_conflict":  suf_conflict,
        "suffix_match":     suf_match,
        "is_non_latin":     non_lat,
        # Address
        "numeric_jaccard":    _numj(s1_nums, cnd_nums),
        "numeric_overlap":    _numovl(s1_nums, cnd_nums),
        "addr_token_jaccard": _tokj(s1_addr, cnd_addr),
        "addr_char3_jaccard": _c3j(s1_addr, cnd_addr),
        "zip_match":          _zipm(s1_nums, cnd_nums),
        "s1_addr_empty":  (s1_addr  == "").astype(np.int8),
        "cnd_addr_empty": (cnd_addr == "").astype(np.int8),
        # Blocking pass-through
        "blk_name_score": name_score,
        "blk_addr_score": addr_score,
        "blk_num_match":  blk_num,
        # Source
        "is_s2": is_s2,
        # Combined
        "combined_score": combined,
    }, index=candidates.index)

    # Context features
    feat["_s1"] = df["source1_entity_id"].values
    feat["cand_rank"] = (
        feat.groupby("_s1")["combined_score"]
             .rank(ascending=False, method="first"))
    max_sc = feat.groupby("_s1")["combined_score"].transform("max")
    feat["score_gap_to_rank1"] = max_sc - feat["combined_score"]
    feat["n_candidates"]        = feat.groupby("_s1")["combined_score"].transform("count")
    feat = feat.drop(columns=["_s1"])

    return feat


FEATURE_COLS = [
    "jaro_winkler", "levenshtein", "token_set_ratio", "token_sort_ratio",
    "char3_jaccard", "char4_jaccard", "token_jaccard", "first_token_eq",
    "suffix_conflict", "suffix_match", "is_non_latin",
    "numeric_jaccard", "numeric_overlap", "addr_token_jaccard",
    "addr_char3_jaccard", "zip_match", "s1_addr_empty", "cnd_addr_empty",
    "blk_name_score", "blk_addr_score", "blk_num_match",
    "is_s2",
    "combined_score", "cand_rank", "score_gap_to_rank1", "n_candidates",
]

In [ ]:
%%writefile src/model.py
"""
LightGBM classifier for entity matching.

Training strategy:
- Positives: pairs from ground truth
- Negatives: blocking candidates that are NOT in ground truth (hard negatives)
- GroupKFold by source1_entity_id so the same entity never straddles train/val
- Threshold tuned directly on out-of-fold macro F0.5 (not on log-loss)
"""

import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(__file__)))

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
import pickle

from src.features import FEATURE_COLS


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _macro_f05(y_true: np.ndarray, y_pred_bin: np.ndarray,
               groups: np.ndarray) -> float:
    """Macro F0.5 — computed per S1 entity then averaged.
    Singletons (no true matches in the group) score 1.0 when y_pred_bin is
    all-zero for that group, 0.0 otherwise."""
    from evaluate import f_beta_score
    group_ids = np.unique(groups)
    scores = []
    for gid in group_ids:
        mask = groups == gid
        yt = y_true[mask]
        yp = y_pred_bin[mask]
        scores.append(f_beta_score(yt, yp, beta=0.5))
    return float(np.mean(scores))


def _tune_threshold(probas: np.ndarray, y_true: np.ndarray,
                    groups: np.ndarray,
                    thresholds=None) -> float:
    """
    Grid-search threshold on OOF predictions.
    Returns threshold that maximises macro F0.5.
    """
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 91)

    best_t, best_f = 0.5, -1.0
    for t in thresholds:
        f = _macro_f05(y_true, (probas >= t).astype(int), groups)
        if f > best_f:
            best_f = f
            best_t = t

    print(f"  Best threshold={best_t:.3f}  F0.5={best_f:.4f}")
    return float(best_t)


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

LGB_PARAMS = dict(
    objective="binary",
    metric="binary_logloss",
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    reg_alpha=0.1,
    reg_lambda=0.1,
    verbose=-1,
    n_jobs=-1,
)


def train_model(
    feat_df: pd.DataFrame,
    labels: np.ndarray,
    groups: np.ndarray,
    n_splits: int = 5,
    neg_sample_ratio: int = 10,
    verbose: bool = True,
):
    """
    Train LightGBM with GroupKFold CV.

    Parameters
    ----------
    feat_df  : feature DataFrame (rows = pairs)
    labels   : 1 = match, 0 = non-match
    groups   : source1_entity_id per row (for GroupKFold)
    neg_sample_ratio : max negatives per positive to use in training

    Returns
    -------
    model       : trained LGBMClassifier on full data
    oof_probas  : out-of-fold predicted probabilities
    threshold   : F0.5-optimal threshold calibrated on OOF
    """
    X = feat_df[FEATURE_COLS].values.astype(np.float32)
    y = labels.astype(np.int32)

    oof_probas = np.full(len(y), np.nan)
    gkf = GroupKFold(n_splits=n_splits)

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        if verbose:
            print(f"  Fold {fold+1}/{n_splits} — "
                  f"train={len(train_idx):,}  val={len(val_idx):,}")

        X_tr, y_tr = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        # Downsample negatives in training fold only
        pos_idx = np.where(y_tr == 1)[0]
        neg_idx = np.where(y_tr == 0)[0]
        max_neg = len(pos_idx) * neg_sample_ratio
        if len(neg_idx) > max_neg:
            rng = np.random.default_rng(42 + fold)
            neg_idx = rng.choice(neg_idx, size=max_neg, replace=False)
        keep = np.concatenate([pos_idx, neg_idx])
        keep.sort()
        X_tr, y_tr = X_tr[keep], y_tr[keep]

        pos_weight = len(y_tr[y_tr == 0]) / max(len(y_tr[y_tr == 1]), 1)
        model = lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=pos_weight)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=0)],
        )
        oof_probas[val_idx] = model.predict_proba(X_val)[:, 1]

    # Tune threshold on OOF predictions
    threshold = _tune_threshold(oof_probas, y, groups)

    # Retrain on full data with best n_estimators from last fold
    if verbose:
        print("  Retraining on full data\u2026")
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    max_neg = len(pos_idx) * neg_sample_ratio
    if len(neg_idx) > max_neg:
        rng = np.random.default_rng(42)
        neg_idx = rng.choice(neg_idx, size=max_neg, replace=False)
    keep = np.concatenate([pos_idx, neg_idx])
    keep.sort()
    X_full, y_full = X[keep], y[keep]

    pos_weight = len(y_full[y_full == 0]) / max(len(y_full[y_full == 1]), 1)
    final_model = lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=pos_weight)
    final_model.fit(X_full, y_full)

    return final_model, oof_probas, threshold


def predict_proba(model, feat_df: pd.DataFrame) -> np.ndarray:
    X = feat_df[FEATURE_COLS].values.astype(np.float32)
    return model.predict_proba(X)[:, 1]


def save_model(model, threshold: float, path: str):
    with open(path, "wb") as f:
        pickle.dump({"model": model, "threshold": threshold}, f)


def load_model(path: str):
    with open(path, "rb") as f:
        obj = pickle.load(f)
    return obj["model"], obj["threshold"]

In [ ]:
%%writefile src/decide.py
"""
Decision layer: converts per-pair probabilities → final matching_results.tsv.

Steps:
1. Each S2/S3 record can be assigned to at most one S1 (one-to-one from the
   candidate side — confirmed by profiling). So we can just threshold
   per-pair probabilities; no global assignment solver is needed.
2. Singleton gate: if all probabilities for an S1 are below `t_singleton`,
   predict empty (which scores 1.0 for a true singleton).
3. Per-entity expected-F0.5 selection: for each S1, sort candidates by
   probability and keep the prefix that maximises expected F0.5.
"""

import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(__file__)))

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Expected F0.5 per entity
# ---------------------------------------------------------------------------

def _expected_f05(proba_sorted: np.ndarray, beta: float = 0.5) -> np.ndarray:
    """
    Given probabilities sorted descending, compute for each prefix length m
    the expected F_beta score if we predict exactly those m candidates.

    E[F_beta | predict top-m] = sum over subsets, but here we use the simple
    greedy approximation: treat probabilities as independent Bernoulli and
    compute E[TP], E[FP], E[FN] for each prefix.

    For each prefix of length m:
        E[TP]  = sum(p_i) for i in 0..m-1
        E[FP]  = m - E[TP]
        E[FN]  = sum(p_i) for i in m..n-1
        E[Prec] = E[TP] / m  (handle m=0 separately)
        E[Rec]  = E[TP] / (E[TP] + E[FN])
    Returns array of expected F_beta for prefix lengths 0..n.
    """
    n = len(proba_sorted)
    cum_tp = np.concatenate([[0.0], np.cumsum(proba_sorted)])
    total_tp = cum_tp[-1]

    scores = np.zeros(n + 1)
    # m=0: predict empty
    # F_beta(empty) = 1 if no true positives (singleton), else 0
    # We can't know that here, so score 0 for non-empty estimator
    # The caller handles the singleton gate separately.
    scores[0] = 0.0

    for m in range(1, n + 1):
        e_tp  = cum_tp[m]
        e_fp  = m - e_tp
        e_fn  = total_tp - e_tp
        denom_prec = m
        denom_rec  = e_tp + e_fn
        if denom_prec == 0 or denom_rec == 0:
            scores[m] = 0.0
            continue
        prec  = e_tp / denom_prec
        rec   = e_tp / denom_rec
        b2    = beta ** 2
        num   = (1 + b2) * prec * rec
        den   = b2 * prec + rec
        scores[m] = num / den if den > 0 else 0.0

    return scores


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def make_predictions(
    candidates: pd.DataFrame,
    probas: np.ndarray,
    all_s1_ids,
    threshold: float = 0.5,
    t_singleton: float = 0.2,
    beta: float = 0.5,
    use_expected_f05: bool = True,
) -> pd.DataFrame:
    """
    Parameters
    ----------
    candidates   : DataFrame with source1_entity_id, candidate_id
    probas       : predicted probability per row (aligned with candidates)
    all_s1_ids   : iterable of ALL S1 entity IDs — every one must appear in output
    threshold    : base probability cutoff
    t_singleton  : if best_prob < t_singleton for an S1, predict empty
    use_expected_f05 : if True, use per-entity expected-F0.5 prefix selection
                       instead of flat threshold

    Returns
    -------
    DataFrame with columns source1_entity_id, matched_entity_ids
    (one row per S1 entity, matched_entity_ids may be empty string)
    """
    cands = candidates.copy()
    cands["proba"] = probas

    results = {}

    # Group by S1
    for s1_id, grp in cands.groupby("source1_entity_id"):
        grp = grp.sort_values("proba", ascending=False).reset_index(drop=True)
        best_prob = grp["proba"].iloc[0]

        # Singleton gate
        if best_prob < t_singleton:
            results[s1_id] = []
            continue

        if use_expected_f05:
            proba_arr = grp["proba"].values
            ef_scores = _expected_f05(proba_arr, beta=beta)
            best_m = int(np.argmax(ef_scores))
            if best_m == 0 or ef_scores[best_m] <= 0:
                results[s1_id] = []
            else:
                results[s1_id] = list(grp["candidate_id"].iloc[:best_m])
        else:
            # Flat threshold
            matched = grp[grp["proba"] >= threshold]["candidate_id"].tolist()
            results[s1_id] = matched

    # Build output DataFrame — every S1 must appear
    rows = []
    for s1_id in all_s1_ids:
        matched = results.get(s1_id, [])
        rows.append({
            "source1_entity_id": s1_id,
            "matched_entity_ids": ",".join(str(m) for m in matched),
        })

    return pd.DataFrame(rows)

In [ ]:
%%writefile evaluate.py
"""
Evaluation utilities.

f_beta_score   — per-entity F_beta score (handles singletons correctly)
macro_f_beta   — macro-average over all S1 entities
blocking_recall — fraction of true matches that appear in candidates
reduction_ratio — how much we reduced the search space
"""

import numpy as np
import pandas as pd


def f_beta_score(y_true: np.ndarray, y_pred: np.ndarray, beta: float = 0.5) -> float:
    """
    Per-entity F_beta.
    y_true / y_pred : 0/1 arrays for a single S1 entity.
    Singleton (y_true all-zero): returns 1.0 if y_pred also all-zero, else 0.0.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n_true = y_true.sum()
    n_pred = y_pred.sum()

    if n_true == 0:
        return 1.0 if n_pred == 0 else 0.0

    tp = (y_true & y_pred).sum()
    if n_pred == 0:
        return 0.0

    prec   = tp / n_pred
    rec    = tp / n_true
    b2     = beta ** 2
    denom  = b2 * prec + rec
    if denom == 0:
        return 0.0
    return (1 + b2) * prec * rec / denom


def macro_f_beta(
    pred_df: pd.DataFrame,
    gt_df: pd.DataFrame,
    beta: float = 0.5,
) -> float:
    """
    Macro-average F_beta over all S1 entities.

    pred_df : columns source1_entity_id, matched_entity_ids (comma-sep or empty)
    gt_df   : same format (ground truth)
    """
    def _parse(s):
        if pd.isna(s) or str(s).strip() == "":
            return set()
        return set(x.strip() for x in str(s).split(",") if x.strip())

    gt_map   = {row.source1_entity_id: _parse(row.matched_entity_ids)
                for row in gt_df.itertuples()}
    pred_map = {row.source1_entity_id: _parse(row.matched_entity_ids)
                for row in pred_df.itertuples()}

    all_s1 = set(gt_map) | set(pred_map)
    scores = []
    for s1_id in all_s1:
        true_set = gt_map.get(s1_id, set())
        pred_set = pred_map.get(s1_id, set())
        all_cands = true_set | pred_set
        if not all_cands:
            scores.append(1.0)
            continue
        id_list = sorted(all_cands)
        y_true = np.array([1 if c in true_set else 0 for c in id_list])
        y_pred = np.array([1 if c in pred_set else 0 for c in id_list])
        scores.append(f_beta_score(y_true, y_pred, beta=beta))

    return float(np.mean(scores))


def blocking_recall(
    candidates: pd.DataFrame,
    gt_df: pd.DataFrame,
) -> float:
    """
    What fraction of true positive pairs appear in the candidate set?
    candidates : columns source1_entity_id, candidate_id
    gt_df      : columns source1_entity_id, matched_entity_ids
    """
    def _parse(s):
        if pd.isna(s) or str(s).strip() == "":
            return []
        return [x.strip() for x in str(s).split(",") if x.strip()]

    cand_set = set(zip(candidates["source1_entity_id"], candidates["candidate_id"]))

    total = found = 0
    for row in gt_df.itertuples():
        for mid in _parse(row.matched_entity_ids):
            total += 1
            if (row.source1_entity_id, mid) in cand_set:
                found += 1

    return found / total if total > 0 else 1.0


def reduction_ratio(
    candidates: pd.DataFrame,
    s1_df: pd.DataFrame,
    s23_df: pd.DataFrame,
) -> float:
    """
    1 - (num_candidate_pairs / total_possible_pairs).
    Higher is better (we eliminated more pairs).
    """
    n_s1  = len(s1_df)
    n_s23 = len(s23_df)
    total_possible = n_s1 * n_s23
    if total_possible == 0:
        return 0.0
    return 1.0 - len(candidates) / total_possible

In [ ]:
%%writefile run_pipeline.py
"""
End-to-end entity resolution pipeline.

Memory strategy:
  - Normalize each file ONE AT A TIME → save to parquet cache → free RAM
  - Blocking loads only the columns it needs (5 cols instead of 8)
  - Features loads all cols from parquet (not re-normalized from scratch)
  - Train data fully freed before test data is loaded

Timeline estimate (Kaggle 16GB RAM, 4 CPU):
  - Normalize + cache 6 files:    ~10 min
  - Block 300K train S1:          ~45 min
  - Feature train (~15M pairs):   ~10 min
  - LightGBM train (5-fold):      ~10 min
  - Block 1.7M test S1:           ~3.5 h
  - Feature test (~85M pairs):    ~45 min
  - Decision + outputs:           ~5 min
  Total:                          ~5.5 hours
"""

import os, sys, time, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

ROOT = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, ROOT)

from src.normalize import normalize_df
from src.blocking  import generate_candidates
from src.features  import compute_features, FEATURE_COLS
from src.model     import train_model, predict_proba, save_model, load_model
from src.decide    import make_predictions
from evaluate      import macro_f_beta, blocking_recall, reduction_ratio

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
def _auto_detect_paths():
    kaggle_input = "/kaggle/input"
    if not os.path.isdir(kaggle_input):
        return None, None
    for root, dirs, files in os.walk(kaggle_input):
        if "train_source1.tsv" in files:
            train_dir = root
            test_dir = None
            for r2, d2, f2 in os.walk(kaggle_input):
                if "test_source1.tsv" in f2:
                    test_dir = r2
                    break
            return train_dir, test_dir
    return None, None

_auto_train, _auto_test = _auto_detect_paths()

DATA_TRAIN = os.environ.get("DATA_TRAIN", _auto_train or "student_resource/dataset/train")
DATA_TEST  = os.environ.get("DATA_TEST",  _auto_test  or "student_resource/dataset/test")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else "output")
MODEL_PATH = os.path.join(OUTPUT_DIR, "model.pkl")
CACHE_DIR  = os.path.join(OUTPUT_DIR, "cache")

TRAIN_SAMPLE_S1  = 300_000
TOP_K_NAME       = 20
TOP_K_ADDR       = 20
NEG_SAMPLE_RATIO = 10
N_FOLDS          = 5
T_SINGLETON      = 0.15
RANDOM_SEED      = 42
CHUNK_SIZE       = 5_000_000

# Columns needed by each step
BLOCK_COLS = ["entity_id", "country", "norm_name", "norm_addr", "numeric_tokens"]
FEAT_COLS  = ["entity_id", "country", "norm_name", "norm_core",
              "norm_suffix", "norm_addr", "numeric_tokens", "is_non_latin"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)


def ts():
    return time.strftime("%H:%M:%S")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def normalize_and_cache(raw_path: str, cache_path: str, tag: str) -> None:
    """Normalize one raw TSV file and save to parquet. Frees RAM immediately."""
    if os.path.exists(cache_path):
        print(f"  [{tag}] Already cached: {cache_path}")
        return
    print(f"[{ts()}] [{tag}] Normalizing {os.path.basename(raw_path)}\u2026")
    t0 = time.time()
    df = pd.read_csv(raw_path, sep="\t")
    df = normalize_df(df)
    df = df.drop(columns=[c for c in ["business_name", "business_address"] if c in df.columns])
    df.to_parquet(cache_path, index=False)
    print(f"  {len(df):,} rows  {time.time()-t0:.0f}s  \u2192 {cache_path}")
    del df
    gc.collect()


def label_candidates(cands, gt_df):
    gt_set = set()
    for row in gt_df.itertuples(index=False):
        if pd.isna(row.matched_entity_ids) or not str(row.matched_entity_ids).strip():
            continue
        for mid in str(row.matched_entity_ids).split(","):
            mid = mid.strip()
            if mid:
                gt_set.add((row.source1_entity_id, mid))
    cands = cands.copy()
    cands["label"] = cands.apply(
        lambda r: int((r["source1_entity_id"], r["candidate_id"]) in gt_set), axis=1
    )
    return cands, gt_set


def compute_features_chunked(cands, s1_df, s23_df, chunk_size=CHUNK_SIZE):
    parts = []
    for start in range(0, len(cands), chunk_size):
        chunk = cands.iloc[start:start + chunk_size]
        parts.append(compute_features(chunk, s1_df, s23_df))
        if (start // chunk_size) % 5 == 0:
            print(f"    chunk {start//chunk_size + 1} ({start:,}/{len(cands):,})", flush=True)
    return pd.concat(parts, ignore_index=True)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    t_total = time.time()

    # ===== 1. Normalize & cache all 6 files (one at a time, free RAM each time) =====
    print(f"\n[{ts()}] === PHASE 1: Normalize & Cache ===")
    norm = {
        "s1_tr": os.path.join(CACHE_DIR, "norm_s1_train.parquet"),
        "s2_tr": os.path.join(CACHE_DIR, "norm_s2_train.parquet"),
        "s3_tr": os.path.join(CACHE_DIR, "norm_s3_train.parquet"),
        "s1_te": os.path.join(CACHE_DIR, "norm_s1_test.parquet"),
        "s2_te": os.path.join(CACHE_DIR, "norm_s2_test.parquet"),
        "s3_te": os.path.join(CACHE_DIR, "norm_s3_test.parquet"),
    }
    normalize_and_cache(f"{DATA_TRAIN}/train_source1.tsv", norm["s1_tr"], "S1-TR")
    normalize_and_cache(f"{DATA_TRAIN}/train_source2.tsv", norm["s2_tr"], "S2-TR")
    normalize_and_cache(f"{DATA_TRAIN}/train_source3.tsv", norm["s3_tr"], "S3-TR")
    normalize_and_cache(f"{DATA_TEST}/test_source1.tsv",   norm["s1_te"], "S1-TE")
    normalize_and_cache(f"{DATA_TEST}/test_source2.tsv",   norm["s2_te"], "S2-TE")
    normalize_and_cache(f"{DATA_TEST}/test_source3.tsv",   norm["s3_te"], "S3-TE")
    print(f"[{ts()}] All files normalized and cached.")

    # ===== 2. Sample train S1 =====
    print(f"\n[{ts()}] === PHASE 2: Train ===")
    gt_full   = pd.read_csv(f"{DATA_TRAIN}/train_ground_truth.tsv", sep="\t")
    s1_tr_all = pd.read_parquet(norm["s1_tr"], columns=["entity_id", "country"])

    rng       = np.random.default_rng(RANDOM_SEED)
    n_sample  = min(TRAIN_SAMPLE_S1, len(s1_tr_all))
    sample_ids = rng.choice(s1_tr_all["entity_id"].values, size=n_sample, replace=False)
    sample_set = set(sample_ids)
    gt_sample  = gt_full[gt_full["source1_entity_id"].isin(sample_set)].reset_index(drop=True)

    val_n   = int(n_sample * 0.1)
    val_ids = set(rng.choice(sample_ids, size=val_n, replace=False))
    del s1_tr_all, gt_full
    gc.collect()

    # Load full S1 train (only blocking cols) filtered to sample
    s1_sample = pd.read_parquet(norm["s1_tr"], columns=FEAT_COLS)
    s1_sample = s1_sample[s1_sample["entity_id"].isin(sample_set)].reset_index(drop=True)
    s1_val    = s1_sample[s1_sample["entity_id"].isin(val_ids)].reset_index(drop=True)
    gt_val    = gt_sample[gt_sample["source1_entity_id"].isin(val_ids)]
    print(f"  Train sample: {len(s1_sample):,} S1  Val: {len(s1_val):,}  GT rows: {len(gt_sample):,}")

    # ===== 3. Blocking train (load s2/s3 with blocking cols only) =====
    cands_cache     = os.path.join(CACHE_DIR, "train_candidates.parquet")
    val_cands_cache = os.path.join(CACHE_DIR, "val_candidates.parquet")

    if os.path.exists(cands_cache):
        print(f"[{ts()}] [BLOCK-TRAIN] Loading from cache\u2026")
        train_cands = pd.read_parquet(cands_cache)
        val_cands   = pd.read_parquet(val_cands_cache)
    else:
        print(f"[{ts()}] [BLOCK-TRAIN] Loading S2/S3 for blocking (slim cols)\u2026")
        s2_tr_blk = pd.read_parquet(norm["s2_tr"], columns=BLOCK_COLS)
        s3_tr_blk = pd.read_parquet(norm["s3_tr"], columns=BLOCK_COLS)

        # Build gt_pairs set for recall-per-pass logging
        gt_pairs_set = set()
        for row in gt_sample.itertuples(index=False):
            if pd.isna(row.matched_entity_ids) or not str(row.matched_entity_ids).strip():
                continue
            for mid in str(row.matched_entity_ids).split(","):
                mid = mid.strip()
                if mid:
                    gt_pairs_set.add((row.source1_entity_id, mid))

        print(f"[{ts()}] [BLOCK-TRAIN] Generating candidates\u2026")
        t0 = time.time()
        train_cands = generate_candidates(
            s1_sample[BLOCK_COLS], s2_tr_blk, s3_tr_blk,
            top_k_name=TOP_K_NAME, top_k_addr=TOP_K_ADDR,
            gt_pairs=gt_pairs_set,
        )
        print(f"  {time.time()-t0:.0f}s  cands={len(train_cands):,}  avg/S1={len(train_cands)/len(s1_sample):.1f}")
        train_cands.to_parquet(cands_cache, index=False)

        print(f"[{ts()}] [BLOCK-VAL] Generating candidates\u2026")
        t0 = time.time()
        val_cands = generate_candidates(
            s1_val[BLOCK_COLS], s2_tr_blk, s3_tr_blk,
            top_k_name=TOP_K_NAME, top_k_addr=TOP_K_ADDR,
        )
        print(f"  {time.time()-t0:.0f}s")
        val_cands.to_parquet(val_cands_cache, index=False)

        del s2_tr_blk, s3_tr_blk
        gc.collect()

    train_cands, _ = label_candidates(train_cands, gt_sample)
    n_pos = train_cands["label"].sum()
    print(f"  positives={n_pos:,}  negatives={(len(train_cands)-n_pos):,}")
    print(f"  Blocking recall: {blocking_recall(train_cands, gt_sample):.4f}")

    # ===== 4. Features train (load s2/s3 with full feat cols) =====
    feats_cache     = os.path.join(CACHE_DIR, "train_features.parquet")
    val_feats_cache = os.path.join(CACHE_DIR, "val_features.parquet")

    if os.path.exists(feats_cache):
        print(f"[{ts()}] [FEAT-TRAIN] Loading from cache\u2026")
        train_feats = pd.read_parquet(feats_cache)
        val_feats   = pd.read_parquet(val_feats_cache)
    else:
        print(f"[{ts()}] [FEAT-TRAIN] Loading S2/S3 for features\u2026")
        s2_tr_ft = pd.read_parquet(norm["s2_tr"], columns=FEAT_COLS)
        s3_tr_ft = pd.read_parquet(norm["s3_tr"], columns=FEAT_COLS)
        s23_tr   = pd.concat([s2_tr_ft, s3_tr_ft], ignore_index=True)
        del s2_tr_ft, s3_tr_ft
        gc.collect()

        print(f"[{ts()}] [FEAT-TRAIN] Computing features\u2026")
        t0 = time.time()
        train_feats = compute_features_chunked(train_cands, s1_sample, s23_tr)
        print(f"  {time.time()-t0:.0f}s  shape={train_feats.shape}")
        train_feats.to_parquet(feats_cache, index=False)

        val_cands, _ = label_candidates(val_cands, gt_val)
        val_feats = compute_features_chunked(val_cands, s1_val, s23_tr)
        val_feats.to_parquet(val_feats_cache, index=False)

        del s23_tr
        gc.collect()

    # ===== 5. Model training =====
    print(f"\n[{ts()}] [MODEL] Training LightGBM ({N_FOLDS}-fold)\u2026")
    t0 = time.time()
    labels = train_cands["label"].values
    groups = train_cands["source1_entity_id"].values
    model, oof_probas, threshold = train_model(
        train_feats, labels, groups,
        n_splits=N_FOLDS, neg_sample_ratio=NEG_SAMPLE_RATIO,
    )
    print(f"  {time.time()-t0:.0f}s  threshold={threshold:.3f}")
    save_model(model, threshold, MODEL_PATH)

    # ===== 6. Validate =====
    val_cands, _ = label_candidates(pd.read_parquet(val_cands_cache), gt_val)
    val_probs = predict_proba(model, val_feats)
    val_pred  = make_predictions(val_cands, val_probs, s1_val["entity_id"].values,
                                 threshold=threshold, t_singleton=T_SINGLETON)
    val_f05   = macro_f_beta(val_pred, gt_val)
    print(f"  Val macro F0.5 = {val_f05:.4f}")

    # ===== Free ALL train data =====
    del train_cands, train_feats, val_cands, val_feats, val_probs, val_pred
    del s1_sample, s1_val
    gc.collect()
    print(f"[{ts()}] Train data freed.")

    # ===== 7. Blocking test (load with slim cols) =====
    print(f"\n[{ts()}] === PHASE 3: Test ===")
    test_cands_cache = os.path.join(CACHE_DIR, "test_candidates.parquet")

    if os.path.exists(test_cands_cache):
        print(f"[{ts()}] [BLOCK-TEST] Loading from cache\u2026")
        test_cands = pd.read_parquet(test_cands_cache)
    else:
        print(f"[{ts()}] [BLOCK-TEST] Loading S1/S2/S3 for blocking\u2026")
        s1_te_blk = pd.read_parquet(norm["s1_te"], columns=BLOCK_COLS)
        s2_te_blk = pd.read_parquet(norm["s2_te"], columns=BLOCK_COLS)
        s3_te_blk = pd.read_parquet(norm["s3_te"], columns=BLOCK_COLS)

        print(f"[{ts()}] [BLOCK-TEST] Generating candidates\u2026")
        t0 = time.time()
        test_cands = generate_candidates(
            s1_te_blk, s2_te_blk, s3_te_blk,
            top_k_name=TOP_K_NAME, top_k_addr=TOP_K_ADDR,
        )
        print(f"  {time.time()-t0:.0f}s  cands={len(test_cands):,}  avg/S1={len(test_cands)/len(s1_te_blk):.1f}")
        test_cands.to_parquet(test_cands_cache, index=False)
        del s2_te_blk, s3_te_blk
        gc.collect()

    # ===== 8. Features test =====
    print(f"\n[{ts()}] [FEAT-TEST] Loading S1/S2/S3 for features\u2026")
    s1_te_ft = pd.read_parquet(norm["s1_te"], columns=FEAT_COLS)
    s2_te_ft = pd.read_parquet(norm["s2_te"], columns=FEAT_COLS)
    s3_te_ft = pd.read_parquet(norm["s3_te"], columns=FEAT_COLS)
    s23_te   = pd.concat([s2_te_ft, s3_te_ft], ignore_index=True)
    del s2_te_ft, s3_te_ft
    gc.collect()

    print(f"[{ts()}] [FEAT-TEST] Computing features ({len(test_cands):,} pairs)\u2026")
    t0 = time.time()
    test_feats = compute_features_chunked(test_cands, s1_te_ft, s23_te)
    print(f"  {time.time()-t0:.0f}s")
    del s23_te
    gc.collect()

    # ===== 9. Inference + decision =====
    print(f"[{ts()}] [INFER] Predicting\u2026")
    test_probs = predict_proba(model, test_feats)

    print(f"[{ts()}] [DECIDE] Applying decision layer\u2026")
    s1_te_ids = pd.read_parquet(norm["s1_te"], columns=["entity_id"])["entity_id"].values
    matching  = make_predictions(
        test_cands, test_probs, s1_te_ids,
        threshold=threshold, t_singleton=T_SINGLETON,
    )

    # ===== 10. Write outputs =====
    matching_path  = os.path.join(OUTPUT_DIR, "matching_results.tsv")
    candidate_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

    matching.to_csv(matching_path, sep="\t", index=False)
    print(f"[{ts()}] Wrote {matching_path}")

    cand_grouped = (
        test_cands.groupby("source1_entity_id")["candidate_id"]
                  .apply(lambda x: ",".join(x.astype(str)))
                  .reset_index()
                  .rename(columns={"candidate_id": "candidate_entity_ids"})
    )
    all_s1_df = pd.DataFrame({"source1_entity_id": s1_te_ids})
    cand_out  = all_s1_df.merge(cand_grouped, on="source1_entity_id", how="left")
    cand_out["candidate_entity_ids"] = cand_out["candidate_entity_ids"].fillna("")
    cand_out.to_csv(candidate_path, sep="\t", index=False)
    print(f"[{ts()}] Wrote {candidate_path}")

    # ===== Summary =====
    n_matched = (matching["matched_entity_ids"].str.strip() != "").sum()
    n_empty   = (matching["matched_entity_ids"].str.strip() == "").sum()
    total_min = (time.time() - t_total) / 60
    print(f"\n=== SUMMARY ===")
    print(f"  Test S1:           {len(matching):,}")
    print(f"  Predicted matched: {n_matched:,}  ({n_matched/len(matching)*100:.1f}%)")
    print(f"  Singletons pred:   {n_empty:,}  ({n_empty/len(matching)*100:.1f}%)")
    print(f"  Val macro F0.5:    {val_f05:.4f}")
    print(f"  Total runtime:     {total_min:.0f} min")

    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
import subprocess, sys, os

def find_data_paths():
    base = "/kaggle/input"
    train_dir, test_dir = None, None
    for root, dirs, files in os.walk(base):
        if "train_source1.tsv" in files:
            train_dir = root
        if "test_source1.tsv" in files:
            test_dir = root
        if train_dir and test_dir:
            break
    return train_dir, test_dir

train_dir, test_dir = find_data_paths()
print(f"Train: {train_dir}")
print(f"Test:  {test_dir}")

env = os.environ.copy()
if train_dir:
    env["DATA_TRAIN"] = train_dir
if test_dir:
    env["DATA_TEST"] = test_dir
env["OUTPUT_DIR"] = "/kaggle/working"

result = subprocess.run([sys.executable, "-u", "run_pipeline.py"], env=env)
print(f"\nExit code: {result.returncode}")

In [ ]:
import os
output_dir = "/kaggle/working"
print("Output files:")
for f in sorted(os.listdir(output_dir)):
    path = os.path.join(output_dir, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {f}  ({size_mb:.2f} MB)")